In [1]:
import os
import glob
import numpy as np
import librosa

In [2]:
# [1] 데이터셋 구성 - TESS + RAVDESS + 감정 필터링

# 데이터셋 불러옴
TESS_path = r'C:\Users\dldpq\kimleepark\TESS Toronto emotional speech set data' # 본인 경로로 수정
RAVDESS_path = r'C:\Users\dldpq\kimleepark\Ravdess_by_emotion'

#사용할 감정 
selected_emotions = ['angry', 'happy', 'neutral', 'sad']


In [3]:
# [2] 데이터셋 분류 - TESS
TESS_files, TESS_labels = [], []

for folder in os.listdir(TESS_path):
    folder_lower= folder.lower()
    for emotion in selected_emotions:
        if emotion in folder_lower:
            emotion_dir = os.path.join(TESS_path, folder)
            for wav in glob.glob(os.path.join(emotion_dir,"*.wav")):
                TESS_files.append(wav)
                TESS_labels.append(emotion)

In [4]:
# [2] 데이터셋 분류 - RAVDESS
RAV_files, RAV_labels = [], []

for emotion in selected_emotions:
    emotion_dir = os.path.join(RAVDESS_path, emotion)
    for wav in glob.glob(os.path.join(emotion_dir,"*.wav")):
        RAV_files.append(wav)
        RAV_labels.append(emotion)

In [5]:
from collections import Counter

# [3] 데이터셋 합치기 - TESS+RAVDESS
all_files = TESS_files + RAV_files
all_labels = TESS_labels + RAV_labels

print("전체 데이터 수 : ", len(all_files))

print("감정별 데이터 수")
counter = Counter(all_labels)
for emo, cnt in counter.items():
    print(f"{emo} : {cnt}")

전체 데이터 수 :  2272
감정별 데이터 수
angry : 592
happy : 592
neutral : 496
sad : 592


In [19]:
#file_path = r"C:\Users\dldpq\kimleepark\TESS Toronto emotional speech set data\OAF_angry\OAF_back_angry.wav"
#y, sr = librosa.load(file_path, sr=None) 
#print("TESS sample rate:", sr)

#file_path = r"C:\Users\dldpq\kimleepark\Ravdess_by_emotion\angry\Actor_01_03-01-05-01-01-01-01.wav"
#y, sr = librosa.load(file_path, sr=None)
#print("RAVDESS sample rate:", sr)


TESS sample rate: 24414
RAVDESS sample rate: 48000


In [6]:
# [4] 데이터 전처리
mfcc_list = []

for file_path in all_files:
    # 1. 오디오 불러오기 (16kHz, 모노)
    y, sr = librosa.load(file_path, sr=16000, mono=True)

    # 2. 길이 고정 (1.6초 = 25600 샘플)
    target_len = 16000 * 16 // 10   # = 25600
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    # 3. MFCC 추출 (40차원, 25ms 창, 10ms 홉)
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=40, n_fft=512,
        hop_length=160, win_length=400
    )  # shape: (40, T)

    # 4. 프레임 수 고정 (160프레임)
    if mfcc.shape[1] < 160:
        pad = np.zeros((40, 160 - mfcc.shape[1]))
        mfcc = np.hstack([mfcc, pad])
    else:
        mfcc = mfcc[:, :160]

    # 5. 정규화 (특징별 평균0, 표준편차1)
    mfcc = (mfcc - mfcc.mean(axis=1, keepdims=True)) / (mfcc.std(axis=1, keepdims=True) + 1e-8)

    mfcc_list.append(mfcc.astype(np.float32))

# 최종 데이터셋
processed_mfcc = np.stack(mfcc_list) 
print("MFCC shape:", processed_mfcc.shape)

MFCC shape: (2272, 40, 160)


In [7]:
from sklearn.preprocessing import LabelEncoder

# [5] 라벨 인코딩 
label_encoder = LabelEncoder()
emotion_labels = label_encoder.fit_transform(all_labels).astype(np.int64)  
print("클래스 인덱스 :", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

클래스 인덱스 : {np.str_('angry'): np.int64(0), np.str_('happy'): np.int64(1), np.str_('neutral'): np.int64(2), np.str_('sad'): np.int64(3)}


In [10]:
from sklearn.model_selection import train_test_split

# [6] 학습/검증 분할
X = np.transpose(processed_mfcc, (0, 2, 1)).astype(np.float32)
y = emotion_labels                                     

print("전체:", X.shape, y.shape)
print("전체 라벨 분포:", Counter(y))


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.15,         
    random_state=42,
    stratify=y             
)
print("\n1차 분할 완료")
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train 라벨 분포:", Counter(y_train))
print("Test  라벨 분포:", Counter(y_test))


X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.15,        
    random_state=42,
    stratify=y_train
)
print("\n최종 세트")
print("Train(final):", X_tr.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Val 라벨 분포:", Counter(y_val))


전체: (2272, 160, 40) (2272,)
전체 라벨 분포: Counter({np.int64(0): 592, np.int64(1): 592, np.int64(3): 592, np.int64(2): 496})

1차 분할 완료
Train: (1931, 160, 40) Test: (341, 160, 40)
Train 라벨 분포: Counter({np.int64(3): 503, np.int64(0): 503, np.int64(1): 503, np.int64(2): 422})
Test  라벨 분포: Counter({np.int64(3): 89, np.int64(0): 89, np.int64(1): 89, np.int64(2): 74})

최종 세트
Train(final): (1641, 160, 40) Val: (290, 160, 40) Test: (341, 160, 40)
Val 라벨 분포: Counter({np.int64(1): 76, np.int64(3): 76, np.int64(0): 75, np.int64(2): 63})


In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models

# [7] Conv1D 모델 정의
num_classes = len(np.unique(y))

model = models.Sequential([
    layers.Input(shape=(160, 40)),                 

    layers.Conv1D(64, kernel_size=5, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPool1D(pool_size=2),                 

    layers.Conv1D(128, kernel_size=5, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPool1D(pool_size=2),                 

    layers.Conv1D(256, kernel_size=3, padding="same", activation="relu"),
    layers.BatchNormalization(),

    layers.GlobalAveragePooling1D(),               
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 160, 64)             │          12,864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 160, 64)             │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 80, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 80, 128)             │          41,088 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 80, 128)             │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 40, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_2 (Conv1D)                    │ (None, 40, 256)             │          98,560 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 40, 256)             │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │             516 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 187,716 (733.27 KB)

 Trainable params: 186,820 (729.77 KB)

 Non-trainable params: 896 (3.50 KB)

In [12]:
# [8] 모델 학습
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
]

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.7727 - loss: 0.5898 - val_accuracy: 0.5621 - val_loss: 0.9333 - learning_rate: 0.0010
Epoch 2/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.8544 - loss: 0.3501 - val_accuracy: 0.8138 - val_loss: 0.4583 - learning_rate: 0.0010
Epoch 3/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.8763 - loss: 0.2895 - val_accuracy: 0.8241 - val_loss: 0.4261 - learning_rate: 0.0010
Epoch 4/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.8952 - loss: 0.2506 - val_accuracy: 0.8276 - val_loss: 0.4172 - learning_rate: 0.0010
Epoch 5/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.9305 - loss: 0.1942 - val_accuracy: 0.8517 - val_loss: 0.4112 - learning_rate: 0.0010
Epoch 6/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.9348 - loss: 0.1596 - val_accuracy: 0.8483 - val_loss: 0.4223 - learning_rate: 0.0010
Epoch 7/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9494 - loss: 0.1300 - val_acc

In [15]:
from sklearn.metrics import classification_report, confusion_matrix

# [9] 성능
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"최종 : loss={test_loss:.4f}, acc={test_acc:.4f}")

y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
print("\n분류 리포트:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

최종 : loss=0.4550, acc=0.8475

분류 리포트:
              precision    recall  f1-score   support

       angry       0.85      0.89      0.87        89
       happy       0.80      0.88      0.84        89
     neutral       0.88      0.77      0.82        74
         sad       0.87      0.84      0.86        89

    accuracy                           0.85       341
   macro avg       0.85      0.84      0.85       341
weighted avg       0.85      0.85      0.85       341



In [16]:
model.save("speech_emotion_model.keras")